# ASTR 457, Day 8 in-class exercise: fitting a double-peaked line, with an assistant

Thu Sep 17. This isn't collected or graded. Work in pairs, 10 minutes, then two pairs defend their answer to the room.

You cannot do this by hand in 10 minutes. That is the point. **UIUC Copilot writes every line of code. You paste the prompts, you run the checks, and you defend the answer.** Each step below has a prompt ready to paste and a list of things to check, because each one has a mistake an assistant makes on this problem more often than not. When a check fails, fix the prompt, not the code. And remember Copilot cannot see your other cells: it guesses variable names, so a `NameError` is your prompt's fault, not its - tell it what exists and what to call things.

The object: **SDSS J233254.46+151305.4** (Strateva et al. 2003, Table 1; SDSS DR17 spectrum 747-52234-354, z = 0.215), a second double-peaked line from the paper we used in lecture. I've included some code below to load the spectrum, divide out the continuum and mask the narrow host-galaxy lines (the same thing we did in lecture), but that's it. Run it first.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats

# --- provided: load, continuum-normalize, mask the narrow host-galaxy lines (the same steps as the lecture) ---
with open('data/sdss_j2332+1513_halpha.csv') as f:
    nhead = sum(1 for line in f if line.startswith('#'))
d = np.genfromtxt('data/sdss_j2332+1513_halpha.csv', delimiter=',', skip_header=nhead, names=True)
wave_all, flux_raw, err_raw = d['wave_rest_A'], d['flux'], d['flux_err']

cw = ((wave_all > 6200) & (wave_all < 6350)) | ((wave_all > 6750) & (wave_all < 6900))
A = np.vstack([np.ones(cw.sum()), wave_all[cw] - 6550]).T * (1 / err_raw[cw])[:, None]
coef, *_ = np.linalg.lstsq(A, flux_raw[cw] / err_raw[cw], rcond=None)
cov_c = np.linalg.inv(A.T @ A)
continuum = coef[0] + coef[1] * (wave_all - 6550)
cont_err = np.sqrt(cov_c[0, 0] + cov_c[1, 1] * (wave_all - 6550)**2 + 2 * cov_c[0, 1] * (wave_all - 6550))
flux_all = flux_raw / continuum
err_all = flux_all * np.sqrt((err_raw / flux_raw)**2 + (cont_err / continuum)**2)

narrow_lines = [6548.0, 6562.8, 6583.5, 6716.4, 6730.8]
use = np.ones_like(wave_all, dtype=bool)
for l in narrow_lines:
    use &= ~((wave_all > l - 8) & (wave_all < l + 8))
wave, flux, err = wave_all[use], flux_all[use], err_all[use]
N = wave.size

def gauss(x, amp, cen, wid):
    return amp * np.exp(-0.5 * ((x - cen) / wid)**2)

def model_k(p, x):
    """1 + a sum of Gaussians; p = [A1, cen1, wid1, A2, cen2, wid2, ...]"""
    m = np.ones_like(x)
    for j in range(0, len(p), 3):
        m = m + gauss(x, *p[j:j+3])
    return m

def chi2_k(p, x=wave, y=flux, s=err):
    for j in range(0, len(p), 3):
        if p[j] < 0 or p[j+2] < 1:
            return 1e30
    return np.sum((y - model_k(p, x))**2 / s**2)

print(f"{wave_all.size} pixels, {N} used after masking; median error {np.median(err):.3f} of the continuum")


465 pixels, 415 used after masking; median error 0.051 of the continuum


## Q1. Plot it, fit one, two and three Gaussians, print $\chi^2$, dof and reduced $\chi^2$ (3 min)

Paste this:

```
In this Jupyter notebook the following already exist: numpy as np, matplotlib.pyplot as plt,
scipy.optimize.minimize, scipy.stats as stats; arrays wave, flux, err (a continuum-normalized
spectrum with the narrow lines masked out), and wave_all, flux_all, err_all (the same before
masking); functions model_k(p, x) = 1 + a sum of Gaussians with p = [A1, cen1, wid1, A2, cen2,
wid2, ...], and chi2_k(p) = the chi-squared of that model against wave, flux, err (it returns
1e30 if any amplitude is negative or any width is below 1).
Write ONE code cell that:
1. plots flux_all vs wave_all with error bars err_all, and shades the masked pixels (those in
   wave_all but not in wave) in grey;
2. fits k = 1, 2 and 3 Gaussians by minimizing chi2_k with scipy.optimize.minimize, method
   Nelder-Mead, options xatol=1e-8, fatol=1e-8, maxiter=40000, with starting guesses you
   choose from the plot: amplitudes ~0.5-1, centers near the visible peaks, widths ~30-60 A;
3. for each k prints chi2, dof = N - 3k where N = wave.size, and reduced chi2 = chi2/dof,
   and the fitted parameters;
4. stores the fits in a dictionary called fits, keyed by k, with the value
   (chi2, best_parameters), so later cells can use them.
Do not redefine model_k or chi2_k. Do not use curve_fit.
```

Check before you believe it: (a) the printed dof is N &minus; 3k, not N; (b) no $\chi^2$ is 1e30 (that means a starting guess broke the bounds); (c) the fitted centers are where the peaks are on the plot, and inside the wavelength range - a center outside the data or a width pinned at 1 means that component is doing nothing; (d) reduced $\chi^2$ falls as k rises; (e) for k = 3, look at the three centers - if two of them are within a few &Aring; of each other, two components are fitting the same peak and the optimizer has stopped somewhere silly. Note the two-Gaussian $\chi^2$ - you'll need it in Q2.


In [2]:
# your work here


In [3]:
# Takeaway for Q1 - run this after your cell; it reads `fits` and tells you what to take from it
try:
    _N = wave.size
    for _k in (1, 2, 3):
        _c2, _p = fits[_k]; _p = np.asarray(_p, float)
        _cen, _wid = _p[1::3], _p[2::3]
        _off = [f"{c:.0f}" for c in _cen if c < wave.min() or c > wave.max()]
        _floor = [f"{w:.2f}" for w in _wid if w < 1.5]
        _dup = any(abs(_cen[i]-_cen[j]) < 10 for i in range(_k) for j in range(i+1, _k))
        _msg = []
        if _off: _msg.append(f"a center at {', '.join(_off)} A is outside the data: that Gaussian is doing nothing")
        if _floor: _msg.append(f"a width of {', '.join(_floor)} A is pinned at the bound: that Gaussian is a spike, not a line")
        if _dup: _msg.append("two centers within 10 A of each other: two components are fitting the same peak")
        print(f"k = {_k}: chi2 {_c2:.1f}, reduced {_c2/(_N-3*_k):.2f}" + ("  <- " + "; ".join(_msg) if _msg else ""))
    _r = [fits[k][0]/(_N-3*k) for k in (1, 2, 3)]
    if not (_r[0] > _r[1] > _r[2]):
        print("\nTakeaway: reduced chi2 did NOT fall every time you added a Gaussian. More parameters can only lower chi2 at the true minimum,\n"
              "so the optimizer stopped somewhere it should not have, and one of your 'fits' is not a fit. Fix the starting guesses in the prompt.")
    elif _r[2] > 1 + 3*np.sqrt(2/(_N-9)):
        print(f"\nTakeaway: every fit got better, and the best is still reduced chi2 = {_r[2]:.2f} when a correct model with these error bars sits\n"
              f"within {np.sqrt(2/(_N-9)):.2f} of 1. Something the model doesn't have is in the data. Keep going: Q2 asks whether these are even the best fits.")
    else:
        print("\nTakeaway: reduced chi2 is near 1, so a sum of Gaussians with your error bars is not obviously wrong yet. Q3 will tell you whether that holds up.")
except NameError as e:
    print(f"Copilot did not store the fits under the name the prompt asked for ({e}). That is the first lesson:\n"
          "it cannot see your other cells, so you have to tell it what to call things. Fix the prompt and re-run.")


Copilot did not store the fits under the name the prompt asked for (name 'fits' is not defined). That is the first lesson:
it cannot see your other cells, so you have to tell it what to call things. Fix the prompt and re-run.


## Q2. Refit two Gaussians from three starting guesses that differ in kind (2 min)

Paste this:

```
Now refit the two-Gaussian model (k = 2) from three starting guesses that are different in
KIND, not three small perturbations of one guess:
  (a) two peaks of similar width at the two visible peaks;
  (b) one broad Gaussian (width ~80 A) centered between the peaks plus one narrow one
      (width ~20 A) on the bluer peak;
  (c) one wide Gaussian (width ~60 A) at the redder peak plus a small bump (amplitude ~0.2,
      width ~25 A) far to the blue.
For each start print the starting guess, the chi2 it reaches, and the fitted centers and
widths. Store the results in a dictionary called results, keyed by the start label, with the
value (chi2, best_parameters). Then print which start reached the lowest chi2.
```

Check: look at the printed starting guesses first - are they what you asked for? Then: do all three reach the same $\chi^2$ AND the same parameters? If the $\chi^2$ values differ by more than a few units (the $\nu = 3$ curve from lecture is your scale), the optimizer stopped in a local minimum on at least one of them. If the $\chi^2$ values agree to within a few units but the parameters do not (a narrow line on the red side in one, on the blue side in another), the data cannot tell those solutions apart, and no single "best fit" exists. Which do you report, and how? If Copilot quietly gave you three versions of guess (a), send it back.


In [4]:
# your work here


In [5]:
# Takeaway for Q2 - run this after your cell; it reads `results`
try:
    _items = [(lab, float(v[0]), np.asarray(v[1], float)) for lab, v in results.items()]
    _c2s = np.array([c for _, c, _ in _items]); _best = _c2s.min()
    _scale = 2*np.sqrt(2*3)   # a few units: two standard deviations of a chi2 with 3 dof
    for _lab, _c2, _p in _items:
        _flag = ""
        if _c2 - _best > _scale: _flag = "  <- stopped in a local minimum"
        if (_p[2::3] < 1.5).any(): _flag += "  <- a width pinned at the bound: that component collapsed into a spike"
        print(f"start {_lab}: chi2 {_c2:.1f}, centers {', '.join(f'{c:.0f}' for c in _p[1::3])}, widths {', '.join(f'{w:.0f}' for w in _p[2::3])}{_flag}")
    _near = [(lab, p) for lab, c2, p in _items if c2 - _best <= _scale]
    _same = all(np.allclose(np.sort(p[1::3]), np.sort(_near[0][1][1::3]), atol=5) for _, p in _near)
    try:
        _q1 = float(fits[2][0])
        if _q1 - _best > _scale:
            print(f"\nAnd look back at Q1: its two-Gaussian fit stopped at chi2 {_q1:.1f}. That was a local minimum too, and nothing in Q1's printout said so.")
    except NameError:
        pass
    if len(_near) < len(_items):
        print("\nTakeaway: the starts that differ in kind did not all reach the same chi2. An optimizer reports where it stopped, and where it\n"
              "stops depends on where you started it. The worst one is not 'a bad fit', it is a fit that never happened. Keep the lowest.")
    if not _same:
        print("\nTakeaway: two starts reached the same chi2 to within a few units but with DIFFERENT parameters (look at the centers). The data\n"
              "cannot tell those solutions apart. There is no single 'best fit' to report here, only the pair of them and what they have in common.")
    if len(_near) == len(_items) and _same:
        print("\nTakeaway: all three starts agree, so the two-Gaussian minimum is probably global for this object. That is worth saying out loud in\n"
              "your defense, because on the lecture object it was not true.")
except NameError as e:
    print(f"`results` is missing ({e}): Copilot did not store the starts under the name the prompt asked for. Fix the prompt and re-run.")


`results` is missing (name 'results' is not defined): Copilot did not store the starts under the name the prompt asked for. Fix the prompt and re-run.


## Q3. Residuals and a QQ-plot for the best two-Gaussian fit (1 min)

Paste this:

```
Using results from the previous cell (a dictionary keyed by start label with values
(chi2, best_parameters)), take the entry with the lowest chi2 and make a figure with
two panels: left, the normalized residuals (flux - model_k(p, wave)) / err against wave, with
a horizontal line at zero; right, a QQ-plot of those normalized residuals against a standard
normal (scipy.stats.probplot with dist='norm'), with the y = x reference line.
```

Check: the residuals are divided by `err`, not raw; the QQ-plot is of the normalized residuals. Is what's left noise, or structure?


In [6]:
# your work here


In [7]:
# Takeaway for Q3 - run this after your cell; it recomputes the normalized residuals of the best two-Gaussian fit
try:
    _lab = min(results, key=lambda k: results[k][0]); _p = np.asarray(results[_lab][1], float)
    _r = (flux - model_k(_p, wave)) / err
    _ks = stats.kstest(_r, 'norm').pvalue
    _s = np.sign(_r); _runs = np.diff(np.flatnonzero(np.diff(_s))); _run = int(_runs.max()) if _runs.size else 1
    print(f"scatter of the normalized residuals: {np.std(_r):.2f} (1.00 if the model and the error bars were both right)")
    print(f"KS test against N(0,1): p = {_ks:.3f};   longest run of same-sign residuals: {_run} pixels in a row")
    if _ks < 0.05 or _run > 12:
        print("\nTakeaway: what is left after two Gaussians is not noise. A run of that many same-sign residuals is structure the model does not have,\n"
              "and the QQ plot should show it as tails bending off the line. Reduced chi2 told you something was wrong; this tells you where.")
    else:
        print("\nTakeaway: the residuals pass as Gaussian noise. If reduced chi2 is still well above 1, the error bars are the suspect, not the model.")
except NameError as e:
    print(f"Needs `results` from Q2 ({e}).")


Needs `results` from Q2 (name 'results' is not defined).


## Q4. Penalize the parameters: the likelihood-ratio test, then AIC and BIC (2 min)

Paste this:

```
Using fits from Q1 (a dictionary keyed by k with values (chi2, best_parameters)) for k = 1
and 3, and for k = 2 the lowest chi2 in results from Q2, compute:
1. the likelihood-ratio test for 1 -> 2 and 2 -> 3: delta_chi2 = chi2_k - chi2_(k+1), and
   log10 of the p-value from scipy.stats.chi2.logsf(delta_chi2, df=3) / np.log(10), where
   df = 3 is the number of EXTRA parameters. Do not use N - 3k as the degrees of freedom
   here, and do not compute p with .sf and then take log10.
2. AIC = chi2 + 2*(3k) and BIC = chi2 + 3k*ln(N) for k = 1, 2, 3, with N = wave.size.
Print a small table: k, chi2, dof, reduced chi2, AIC, BIC, and the two delta_chi2 / log10 p
lines.
```

Check: (a) the test's degrees of freedom are 3, not N &minus; 3k - assistants get this wrong constantly, even when told; (b) $\log_{10} p$ comes out around &minus;30, not "p = 0.0"; (c) AIC and BIC use k = 3, 6, 9 free parameters, and only differences mean anything.


In [8]:
# your work here


In [9]:
# Takeaway for Q4 - run this after your cell; it recomputes the comparison from `fits` and `results` so it does not depend on Copilot's table
try:
    _N = wave.size
    _c2 = {1: fits[1][0], 2: min(v[0] for v in results.values()), 3: fits[3][0]}
    for _k in (1, 2):
        _d = _c2[_k] - _c2[_k+1]; _lp = stats.chi2.logsf(_d, df=3)/np.log(10)
        print(f"{_k} -> {_k+1}: delta chi2 = {_d:.1f} for 3 extra parameters, log10 p = {_lp:.0f}   (with df = N - 3k instead, p would be {stats.chi2.sf(_d, df=_N-3*(_k+1)):.2f}: the wrong test, and it says the extra line is nothing)")
    _bic = {k: _c2[k] + 3*k*np.log(_N) for k in (1, 2, 3)}
    print("BIC: " + ", ".join(f"k={k}: {v:.0f}" for k, v in _bic.items()) + f"   (delta BIC 2->3 = {_bic[2]-_bic[3]:.0f})")
    _red3 = _c2[3]/(_N-9); _spread = np.sqrt(2/(_N-9))
    print(f"\nTakeaway: every test says three Gaussians beat two, and two beat one, by more than chance could buy. And yet the three-Gaussian fit still has\n"
          f"reduced chi2 = {_red3:.2f}, {(_red3-1)/_spread:.0f} spreads above what a correct model gives. The likelihood-ratio test, AIC and BIC pick the best\n"
          "of the models you gave them. None of them can tell you that a sum of Gaussians is the wrong family. Only the residuals can.")
except NameError as e:
    print(f"Needs `fits` from Q1 and `results` from Q2 ({e}).")


Needs `fits` from Q1 and `results` from Q2 (name 'fits' is not defined).


## Q5. So how many Gaussians is it? (2 min)

Paste this:

```
Copilot cannot see my plots, so first compute two numbers from the normalized residuals of the
best two-Gaussian fit: the p-value of a Kolmogorov-Smirnov test of those residuals against a
standard normal (scipy.stats.kstest), and the longest run of consecutive residuals with the same
sign. Then, using those two numbers, the table and the likelihood-ratio results from the
previous cells, write a verdict in two sentences: is the broad H-alpha line of this AGN best
described by one Gaussian, two, three, or is a sum of Gaussians the wrong model altogether? Cite
the specific numbers (reduced chi2 against its expected spread sqrt(2/dof), delta chi2 and
log10 p, delta AIC, delta BIC, the KS p-value, the longest run). Then state one thing about
these fits that should make me distrust that verdict. Print it as plain text.
```

Then, in a markdown cell, in 1-2 lines of your own: do you agree with it? Say which of its numbers or which plot you'd point a skeptical referee at, and which check above caught something. If every check passed first time, say so - and say which checks you actually ran. Do this before we go over it as a class.


In [10]:
# your work here


In [11]:
# Takeaway for Q5 - the part of the verdict that does not depend on which sentences Copilot wrote
try:
    _N = wave.size; _c2 = {1: fits[1][0], 2: min(v[0] for v in results.values()), 3: fits[3][0]}
    _red = {k: _c2[k]/(_N-3*k) for k in _c2}
    print("Reduced chi2 by k: " + ", ".join(f"{k}: {v:.2f}" for k, v in _red.items()) + f";  a correct model sits within {np.sqrt(2/(_N-9)):.2f} of 1.")
    print("\nTakeaway: Copilot usually answers 'three', because three wins every comparison you asked it to run, or it hedges and names no number\n"
          "at all ('must be interpreted carefully'). Either way it dodged the question. 'Three' is the right answer to the\n"
          "wrong question. The lecture object was a disk, not a sum of Gaussians, and the residuals here say the same. The number to defend is not\n"
          "how many Gaussians, it is how far the best one still is from 1, and what shape is left over. If Copilot's verdict cites a plot it never\n"
          "looked at, or a number it never computed, that is the thing it got wrong.")
except NameError as e:
    print(f"Needs `fits` and `results` ({e}).")


Needs `fits` and `results` (name 'fits' is not defined).


## If you finish early

```
For the two-Gaussian fit with the lowest chi2 in results, estimate the uncertainty on the center
of the second Gaussian two ways: (1) build the Hessian of chi2_k at the best fit by central finite
differences, invert it, and use covariance = 2 * inverse Hessian (because chi2 = -2 ln L),
reporting the standard deviation of cen2 and the correlation coefficient between cen2 and wid2;
(2) a bootstrap: resample the pixels with replacement 50 times and refit k = 2 from the best fit
each time. IMPORTANT: chi2_k(p, x, y, s) took wave, flux, err as DEFAULT arguments when it was
defined, so reassigning those names does nothing; call chi2_k(p, x=wave_bs, y=flux_bs, s=err_bs)
with the resampled arrays explicitly. Store the 50 bootstrap values of cen2 in an array called
cen2_boot, the two standard deviations as sigma_cen2_hess and sigma_cen2_boot, and print both.
```

Check: if the bootstrap standard deviation comes out as exactly 0.000, every refit used the original data. That is the default-argument trap in the prompt above, and Copilot falls into it even when told. Then compare the two numbers and comment on the difference. If there is still time: 1.4826 &times; MAD of the normalized residuals (Day 3), a 3$\sigma$ clip and a refit: outliers, or the wrong model?


In [12]:
# your work here


In [13]:
# Takeaway for the extra - run this after your cell; it reads `cen2_boot` and, if present, `sigma_cen2_hess`
try:
    _lab = min(results, key=lambda k: results[k][0]); _p = np.asarray(results[_lab][1], float)
    print(f"'cen2' here is the component at {_p[4]:.0f} A (width {_p[5]:.0f} A): the {'narrow' if _p[5] < _p[2] else 'broad'} one")
    _sb = float(np.std(np.asarray(cen2_boot, float), ddof=1))
    if _sb < 1e-6:
        print(f"bootstrap sigma(cen2) = {_sb:.6f}\n\nTakeaway: fifty resamples, zero scatter. Copilot resampled the data and then fit the original data fifty times,\n"
              "because chi2_k captured wave, flux and err as default arguments the moment it was defined. Reassigning the global names\n"
              "does nothing to it. A number that is exactly zero is never a measurement; it is a bug telling you where to look.")
    else:
        try:
            _sh = float(sigma_cen2_hess)
            print(f"curvature sigma(cen2) = {_sh:.2f} A,  bootstrap sigma(cen2) = {_sb:.2f} A")
            if _sb > 1.2*_sh:
                print("\nTakeaway: the bootstrap is the larger one. It sees the structure left in the residuals; the curvature only sees the shape of chi2\n"
                      "at the minimum and assumes the model is right. When the model is wrong, believe the bootstrap, and say why.")
            elif _sh > 1.2*_sb:
                print("\nTakeaway: the curvature is the larger one. The chi2 surface is flatter along this center than the data scatter implies, which\n"
                      "usually means it is correlated with another parameter (look at the correlation coefficient you printed). Neither number is\n"
                      "trustworthy on its own when the model is wrong; the disagreement is the finding.")
            else:
                print("\nTakeaway: the two agree to within 20%. That is what you expect when the fit is at a real minimum and the errors are honest;\n"
                      "it does not mean the model is right, only that this parameter is well pinned within it.")
        except NameError:
            print(f"bootstrap sigma(cen2) = {_sb:.2f} A (sigma_cen2_hess not stored, so no comparison here: read both off your printout)")
except NameError as e:
    print(f"Needs `results` and `cen2_boot` ({e}): the prompt asked for them by name.")


Needs `results` and `cen2_boot` (name 'results' is not defined): the prompt asked for them by name.
